# H08 — Parameter estimation: statistical error and instrumental effects

**Part VI · the laboratory of lecture 16**

**Data.**

- `labs/data/gw150914_H1.hdf5`: the strain of the LIGO Hanford detector (H1), 32 s around the event GW150914 (GPS 1126259447 to 1126259479, merger at 1126259462.42), sampled at 4096 Hz. The file comes from the Gravitational Wave Open Science Center (GWOSC), release R1. `load_strain` reads it. The code high-passes the strain at 15 Hz and resamples it to 2048 Hz. A1, A3 and B1 to B4 use it.
- A2 and B5 have no strain. The amplitude that one detector measures is computed from a **simulated** source at $r = 440$ Mpc and $\iota = 0.5$ rad, with an uncertainty of 7 %.
- The glitch of A3 and B4 is **simulated**: a sine-Gaussian added to the H1 data. It is a model of what the instrument does and not a recording of it.
- A4 and B6 use 100 **simulated** numbers drawn from a Gaussian of mean 3 and width 4.
- A5 and B7 use the noise of H1 and L1 **simulated** from the design spectra, 4 s at 2048 Hz, with an injected GW150914-like signal. The noise is Gaussian and stationary.

Everything runs offline.

**Libraries.**

- `numpy`: arrays, random numbers, `percentile` and `corrcoef`.
- `matplotlib.pyplot`: all the plots.
- `pycbc`: the time series, the matched filter, the PSD and the waveform `IMRPhenomD` of the templates in A1, A3 and B1 to B4.
- `lal`: the waveform library that `pycbc` calls. The first cell switches off the redirection of its output.
- `bilby`: A4, A5, B6 and B7. It gives the likelihood class, the priors, the nested sampler `dynesty`, the simulated noise, the injection and the corner plot.
- `os` and `sys`: find the folder of the course, so that the two course modules can be imported.
- `gwcourse_signal` and `gwcourse_palette`: the function of the course that loads the strain and the colors.
- `tomllib`: reads `labs/config.toml`, the configuration of the laboratories: the folder of the data, the events and their files.

**What we do.**

- Part A, the demonstration: the likelihood profile in chirp mass, the degeneracy of distance and inclination, a glitch on the merger, a nested sampler on a Gaussian likelihood and on an injected binary.
- Part B, on your own: the effect of the noise spectrum, the calibration, the distance of a glitch, a second detector, the Bayes factor of the Gaussian model and the full configuration of the sampler.

> **References**
>
> (Ashton and others 2019), (Cutler and Flanagan 1994), (LIGO Scientific Collaboration and Virgo Collaboration 2017), (LIGO Scientific and Virgo Collaborations 2019)

## Tasks

**Part A. The demonstration.** We show how a posterior is computed on real data and how the instrument changes it. All the code is written out and runs cell by cell. Under each result we say what it shows. A2 gives the statistical width of the distance and A5 the statistical widths of the masses. The tasks of Part B each give a shift that is compared with these widths.

- **A1.** The likelihood profile in chirp mass: the functions `condition`, `template_for` and `profile`; the peak.
- **A2.** The degeneracy of distance and inclination: the posterior sampled with a Metropolis chain, for two priors of the distance.
- **A3.** A glitch on the merger: the profile with a weak glitch, with a loud glitch and after the subtraction.
- **A4.** A nested sampler on a Gaussian likelihood with a known answer: the posterior of $\mu$ and $\sigma$ and the evidence.
- **A5.** Nested sampling on an injected binary in simulated noise: three free parameters, the light configuration of the sampler and the corner plot.

**Part B. Your turn, at home.** The functions of Part A are given, with one more. We use them with our own choices.

- **B1.** The estimate of the noise spectrum: the shift of $\Phi$ with the length of the Welch segments.
- **B2.** The noise spectrum from a stretch of data far from the event.
- **B3.** The calibration: the shift of $\Phi$ and of the distance for an amplitude error of $\pm 5$ %.
- **B4.** How far from the merger a glitch can be and be ignored.
- **B5.** A second detector and the degeneracy of distance and inclination.
- **B6.** The Gaussian model with a fixed $\sigma$ and the Bayes factor.
- **B7.** Optional: the full configuration of the sampler, compared with the light one.

In [ ]:
# libraries
import os
import sys
import numpy as np
import matplotlib.pyplot as plt

# LAL redirects its printed output inside Jupyter and every waveform call becomes a hundred times slower: switch it off
import warnings
warnings.filterwarnings("ignore", "Wswiglal-redir-stdio")
import lal
lal.swig_redirect_standard_output_error(False)

# the course root holds gwcourse_palette.py: find it from wherever we are
here = os.path.abspath('.')
while not os.path.exists(os.path.join(here, 'gwcourse_palette.py')):
    if os.path.dirname(here) == here:
        raise FileNotFoundError('gwcourse_palette.py not found: open the notebook from the course folder')
    here = os.path.dirname(here)
sys.path.insert(0, here)
sys.path.insert(0, os.path.join(here, 'labs'))

# the configuration of the laboratories: where the data are and the events we use
import tomllib
with open(os.path.join(here, 'labs', 'config.toml'), 'rb') as config_file:
    config = tomllib.load(config_file)
DATA_DIR = os.path.join(here, 'labs', config['data']['dir'])
GPS_GW150914 = config['events']['GW150914']['gps']
H1_FILE = os.path.join(DATA_DIR, config['events']['GW150914']['H1'])

# the course colors and the course function that reads the strain
from gwcourse_palette import P, apply_style
from gwcourse_signal import load_strain
apply_style()

# the pycbc tools: time series, filter, PSD and waveforms
from pycbc.types import TimeSeries
from pycbc.filter import matched_filter, highpass, resample_to_delta_t
from pycbc.psd import interpolate
from pycbc.waveform import get_td_waveform

# bilby, for the nested sampler of A4, A5, B6 and B7
import bilby

## A1. The likelihood profile in chirp mass

Lecture 16: the log likelihood of the parameters $\boldsymbol{\theta}$ for the data $s$ and the template $h(\boldsymbol{\theta})$ is

$$
\ln P(s\mid\boldsymbol{\theta}) = \langle s, h(\boldsymbol{\theta})\rangle
- \tfrac{1}{2}\langle h(\boldsymbol{\theta}), h(\boldsymbol{\theta})\rangle + \text{const},
$$

with $\langle a, b\rangle$ the inner product weighted by the one-sided PSD $S_{n}(f)$. Maximized over the amplitude and the phase of the template it becomes $\tfrac{1}{2}\Phi^{2}$, with $\Phi$ the output of the matched filter of lecture 13.

We scan the chirp mass $\mathcal{M}$ in the detector frame. For each value we filter the data with an equal mass template and keep the largest $|\Phi|$ within 0.1 s of the event. The curve of $\tfrac{1}{2}\Phi^{2}$ against $\mathcal{M}$ is the profile of the likelihood.

Four functions are used in all the sections. `load_h1(path, f_high_pass=15.0, fs_out=2048.0)` reads the strain and returns it as a `pycbc` time series. `condition(series, edge=2, seglen=4)` removes `edge` seconds from each end and estimates $S_{n}(f)$ with Welch's method on segments of `seglen` seconds. `template_for(chirp_mass, series, f_low=20.0)` returns an equal mass template of a given chirp mass. `profile(series, psd, grid, ...)` returns the largest $|\Phi|$ near the event for each chirp mass of the grid.

In [ ]:
def load_h1(path=H1_FILE, f_high_pass=15.0, fs_out=2048.0):
    """Read the strain of H1, high-pass it and resample it.

    Args:
        path: the file with the strain (default: the H1 file of GW150914).
        f_high_pass: the frequency below which the strain is removed (Hz, default 15).
        fs_out: the sampling frequency after the resampling (Hz, default 2048).

    Returns:
        resampled: the strain, a pycbc TimeSeries.
    """
    # the strain of the file as a pycbc time series
    strain, times, fs = load_strain(path)
    series = TimeSeries(strain, delta_t=1.0 / fs, epoch=times[0])
    # remove the slow drift, then resample
    filtered = highpass(series, f_high_pass)
    resampled = resample_to_delta_t(filtered, 1.0 / fs_out)
    return resampled


def condition(series, edge=2, seglen=4):
    """Crop the edges of the data and estimate the noise spectrum.

    Args:
        series: the data, a pycbc TimeSeries.
        edge: the time cropped at each end (s, default 2).
        seglen: the length of the Welch segments (s, default 4).

    Returns:
        d: the cropped data.
        psd: the one-sided PSD S_n(f) on the frequency grid of d.
    """
    # crop the ends, where the filters leave transients
    d = series.crop(edge, edge)
    # Welch estimate on segments of seglen seconds, then on the frequency grid of the data
    psd = interpolate(d.psd(seglen), d.delta_f)
    return d, psd


def template_for(chirp_mass, series, f_low=20.0):
    """Build an equal mass template of a given chirp mass.

    Args:
        chirp_mass: the chirp mass (solar masses, detector frame).
        series: the data, a pycbc TimeSeries. The template has its length and its sampling.
        f_low: the lower edge of the band (Hz, default 20).

    Returns:
        template: the template at 1 Mpc, as long as the data, with the merger at time zero.
    """
    # two equal masses m have the chirp mass m / 2**(1/5)
    component = chirp_mass * 2 ** 0.2
    # the waveform at 1 Mpc, made as long as the data
    hp, _ = get_td_waveform(approximant='IMRPhenomD', mass1=component, mass2=component,
                            delta_t=series.delta_t, f_lower=f_low)
    hp.resize(len(series))
    # the merger at time zero
    template = hp.cyclic_time_shift(hp.start_time)
    return template


def profile(series, psd, grid, t_event=GPS_GW150914, half=0.1, f_low=20.0, crop=(6, 4)):
    """Find the largest |Phi| near the event for each chirp mass of a grid.

    Args:
        series: the data, a pycbc TimeSeries.
        psd: the one-sided PSD S_n(f) of the noise.
        grid: the chirp masses (solar masses).
        t_event: the GPS time of the event (s, default the merger of GW150914).
        half: the half width of the window around the event (s, default 0.1).
        f_low: the lower edge of the band (Hz, default 20).
        crop: the time cropped at the start and at the end of the output (s, default 6 and 4).

    Returns:
        phi_max: the largest |Phi| in the window, one value for each chirp mass of the grid.
    """
    phi_max = np.zeros(len(grid))
    for k in range(len(grid)):
        # the filter output for the template of this chirp mass, without the transients at the edges
        template = template_for(grid[k], series, f_low=f_low)
        phi = matched_filter(template, series, psd=psd, low_frequency_cutoff=f_low)
        phi = phi.crop(crop[0], crop[1])
        # the largest |Phi| within half seconds of the event
        t_axis = phi.sample_times.numpy()
        near = np.abs(t_axis - t_event) < half
        phi_max[k] = np.max(np.abs(np.asarray(phi))[near])
    return phi_max


# the strain of H1, the cropped data and the noise spectrum
raw = load_h1(path=H1_FILE)
data, psd = condition(raw, edge=2, seglen=4)
print(f"{raw.duration} s of H1 at {1 / raw.delta_t:.0f} Hz, the event at GPS {GPS_GW150914:.2f}; "
      f"{data.duration} s left after the crop")

# the profile: chirp masses from 24 to 40 solar masses, the largest |Phi| for each
grid = np.arange(24.0, 40.01, 0.5)
phi_profile = profile(data, psd, grid=grid)

# the log likelihood, Phi^2 / 2, relative to its maximum, and the peak
log_like = 0.5 * phi_profile ** 2
log_like_rel = log_like - log_like.max()
i_peak = np.argmax(phi_profile)
peak_mass = grid[i_peak]
print(f"peak at {peak_mass:.1f} solar masses, Phi = {phi_profile.max():.1f}")
print(f"ln P relative to the peak: {log_like_rel[i_peak - 1]:.1f} at {grid[i_peak - 1]:.1f}, "
      f"{log_like_rel[i_peak + 1]:.1f} at {grid[i_peak + 1]:.1f}")

# the profile, with the numbers in a box in the empty upper right of the plot
plt.figure(figsize=(7.2, 4.2))
plt.plot(grid, log_like_rel, color=P['BLUE'], lw=2.0)
plt.xlabel(r'chirp mass  [$M_\odot$], detector frame')
plt.ylabel(r'$\ln P(s\mid\theta) - \ln P_{\max}$')
plt.ylim(-40, 2)
summary = (f"peak: {peak_mass:.1f} $M_\\odot$\n"
           f"$\\Phi$ at the peak: {phi_profile.max():.1f}\n"
           f"one step away: {log_like_rel[i_peak - 1]:.1f} and {log_like_rel[i_peak + 1]:.1f}")
plt.text(0.97, 0.95, summary, transform=plt.gca().transAxes, ha='right', va='top', fontsize=10,
         bbox=dict(boxstyle='round', facecolor='white', edgecolor=P['GREY_MID']))
plt.show()

**What the result shows.** The peak is at 32.0 solar masses with $\Phi = 19.5$. The log likelihood falls by 0.2 at 31.5 and by 1.1 at 32.5: the peak is about one solar mass wide. The phase measures the chirp mass best, as in lecture 16.

Two limits of the result. The templates are equal mass and GW150914 is not, so the peak is 1.6 solar masses above the published detector frame value, 30.4, which is inside the published interval. And this is a profile of the likelihood and not a posterior: nothing is marginalized and no prior is applied.

## A2. The degeneracy of distance and inclination

One detector measures one amplitude. The amplitude contains the distance $r$ and the inclination $\iota$ in a single combination:

$$
A \;\propto\; \frac{g(\iota)}{r},
\qquad
g(\iota) = \sqrt{\left(\frac{1+\cos^{2}\iota}{2}\right)^{2} + \cos^{2}\iota}
$$

$g(\iota)$ is the inclination factor of lecture 16. It is $\sqrt{2}$ face on and $1/2$ edge on. Here the detector responds to both polarizations with the same weight. The function `signal_amplitude(distance, iota, f_plus=1.0, f_cross=1.0)` gives $A$ up to a constant. The arguments `f_plus` and `f_cross` are the responses of the detector to the two polarizations. With the default 1 and 1 it is the formula above. B5 uses other values.

We sample the posterior $P(r,\iota\mid A)$ for a measured amplitude with an uncertainty of 7 %, with a Metropolis chain of lecture 16. The prior of the inclination is isotropic: $\cos\iota$ is uniform. The prior of the distance is $p(r) \propto r^{2}$ in one run and flat in the other. `log_posterior` is the logarithm of the posterior up to a constant. `sample` runs the chain. The measured amplitude is computed from a source at 440 Mpc and 0.5 rad, so we know the answer.

We plot the samples in the plane of $r$ against $\cos\iota$ and report the median and the 90 % interval of $r$ for each prior.

In [ ]:
# the measured amplitude is computed from a source at 440 Mpc, inclination 0.5 rad, with a 7 % error
R_TRUE = 440.0
IOTA_TRUE = 0.5
FRACTIONAL_ERROR = 0.07


def signal_amplitude(distance, iota, f_plus=1.0, f_cross=1.0):
    """Compute the amplitude that a detector measures, up to a constant.

    Args:
        distance: the distance r of the source (Mpc).
        iota: the inclination of the orbit (rad).
        f_plus: the response of the detector to the plus polarization (default 1).
        f_cross: the response of the detector to the cross polarization (default 1).

    Returns:
        amp: the amplitude, in arbitrary units. For the default responses it is g(iota) / r.
    """
    # the two polarizations: plus is weighted by (1 + cos^2) / 2 and cross by cos
    c = np.cos(iota)
    plus_part = f_plus * (1 + c ** 2) / 2
    cross_part = f_cross * c
    # they add in quadrature, and the amplitude falls as 1 / r
    amp = np.sqrt(plus_part ** 2 + cross_part ** 2) / distance
    return amp


def make_detector(f_plus=1.0, f_cross=1.0, distance=R_TRUE, iota=IOTA_TRUE):
    """Describe a detector by its responses and the amplitude it measures from the source.

    Args:
        f_plus: the response of the detector to the plus polarization (default 1).
        f_cross: the response of the detector to the cross polarization (default 1).
        distance: the distance of the source that is measured (Mpc, default 440).
        iota: the inclination of the source that is measured (rad, default 0.5).

    Returns:
        detector: the tuple (measured amplitude, f_plus, f_cross).
    """
    a_obs = signal_amplitude(distance, iota, f_plus=f_plus, f_cross=f_cross)
    detector = (a_obs, f_plus, f_cross)
    return detector


# the detector of this section: both responses 1, the amplitude of the source at 440 Mpc
ONE_DETECTOR = [make_detector(f_plus=1.0, f_cross=1.0)]


def log_posterior(distance, iota, prior='volume', detectors=ONE_DETECTOR, error=FRACTIONAL_ERROR):
    """Compute the logarithm of the posterior of (r, iota), up to a constant.

    Args:
        distance: the distance r (Mpc, allowed from 50 to 2000).
        iota: the inclination (rad, allowed from 0 to pi / 2).
        prior: 'volume' for p(r) proportional to r^2, 'flat' for a flat p(r) (default 'volume').
        detectors: a list of tuples (measured amplitude, f_plus, f_cross), one for each detector (default ONE_DETECTOR).
        error: the fractional uncertainty of each amplitude (default 0.07).

    Returns:
        log_post: ln P(r, iota | A) + const, minus infinity outside the allowed ranges.
    """
    # outside the allowed ranges the posterior is zero
    if not 50 < distance < 2000 or not 0 < iota < np.pi / 2:
        return -np.inf
    # the Gaussian likelihood of each measured amplitude, added over the detectors
    log_like = 0.0
    for a_obs, f_plus, f_cross in detectors:
        model = signal_amplitude(distance, iota, f_plus=f_plus, f_cross=f_cross)
        residual = (model - a_obs) / (error * a_obs)
        log_like = log_like - 0.5 * residual ** 2
    # the prior of the distance: r^2 for sources uniform in volume, or flat
    if prior == 'volume':
        log_prior_r = 2 * np.log(distance)
    else:
        log_prior_r = 0.0
    # the prior of the inclination: cos(iota) uniform, which is sin(iota) in iota
    log_prior = log_prior_r + np.log(np.sin(iota))
    log_post = log_like + log_prior
    return log_post


def sample(prior='volume', detectors=ONE_DETECTOR, steps=60000, step_size=0.08, seed=4):
    """Draw samples of (r, iota) with the Metropolis rule.

    Args:
        prior: 'volume' or 'flat', as in log_posterior (default 'volume').
        detectors: the list of detectors of log_posterior (default ONE_DETECTOR).
        steps: the number of steps of the chain (default 60000).
        step_size: the width of a random step, relative in r and in radians in iota (default 0.08).
        seed: the seed of the random numbers, so that the run can be repeated (default 4).

    Returns:
        samples: an array with two columns, r (Mpc) and iota (rad), without the first fifth of the chain.
    """
    rng = np.random.default_rng(seed)
    # the starting point and its posterior
    distance = 400.0
    iota = 0.7
    current = log_posterior(distance, iota, prior=prior, detectors=detectors)
    chain = []
    for _ in range(steps):
        # propose a point: a random step in ln r and in iota
        proposal_d = distance * np.exp(step_size * rng.standard_normal())
        proposal_i = iota + step_size * rng.standard_normal()
        trial = log_posterior(proposal_d, proposal_i, prior=prior, detectors=detectors)
        # the log-normal step is not symmetric, hence the extra term in the acceptance ratio
        log_ratio = trial - current + np.log(proposal_d / distance)
        u = rng.random()
        # accept the point, or stay where we are
        if np.log(u) < log_ratio:
            distance = proposal_d
            iota = proposal_i
            current = trial
        chain.append((distance, iota))
    # drop the first fifth of the chain, the burn-in
    samples = np.array(chain[steps // 5:])
    return samples


# one chain for each prior of the distance
priors_r = ['volume', 'flat']
colors_r = [P['BLUE'], P['BRICK']]
names_r = ['$p(r) \\propto r^2$', '$p(r)$ flat']

plt.figure(figsize=(6.6, 5.0))
summary_lines = []
intervals_r = {}
for prior, color, name in zip(priors_r, colors_r, names_r):
    chain = sample(prior=prior)
    distance = chain[:, 0]
    cos_iota = np.cos(chain[:, 1])
    # every twentieth sample, in the plane of r and cos(iota)
    plt.plot(distance[::20], cos_iota[::20], '.', ms=1.5, color=color, alpha=0.35, label=name)
    # the median and the 90 % interval of r, and its correlation with cos(iota)
    low, med, high = np.percentile(distance, [5, 50, 95])
    correlation = np.corrcoef(distance, cos_iota)[0, 1]
    intervals_r[prior] = (low, med, high)
    print(f"{prior:7s} prior: r = {med:4.0f} +{high - med:3.0f} -{med - low:3.0f} Mpc,"
          f"  correlation with cos(iota) = {correlation:+.2f}")
    summary_lines.append(f"{name}: $r$ = {med:.0f} Mpc, 90 % [{low:.0f}, {high:.0f}]")

plt.xlabel('distance $r$ [Mpc]')
plt.ylabel(r'$\cos\iota$')
plt.legend(frameon=False, markerscale=8, loc='lower center', bbox_to_anchor=(0.5, 1.0), ncol=2)
plt.text(0.97, 0.03, '\n'.join(summary_lines), transform=plt.gca().transAxes, ha='right', va='bottom',
         fontsize=9, bbox=dict(boxstyle='round', facecolor='white', edgecolor=P['GREY_MID']))
plt.show()

**What the result shows.** The samples lie along a curved ridge. The correlation of $r$ with $\cos\iota$ is $+0.94$ for the volume prior and $+0.96$ for the flat prior. A source seen face on is intrinsically louder, so it can be farther away and give the same amplitude. One detector cannot separate the two and the sampler does not change this.

| prior of $r$ | median | 90 % interval | width |
|---|---|---|---|
| $\propto r^{2}$ | 409 Mpc | 211 to 526 Mpc | 314 Mpc |
| flat | 333 Mpc | 179 to 501 Mpc | 322 Mpc |

The median moves by 76 Mpc, 19 % of 409 Mpc, when only the prior changes. This shift comes from an assumption and not from the data. The 90 % interval, about 300 Mpc wide, is the statistical width of the distance for the tasks of Part B. The source is at 440 Mpc and the interval contains it.

## A3. A glitch on the merger

On 17 August 2017 a loud glitch arrived in Livingston 1.1 s before the merger of GW170817. It had to be modeled and subtracted before the parameters could be estimated (LIGO Scientific Collaboration and Virgo Collaboration 2017). We reproduce the problem.

`sine_gaussian(t_arr, t0, f0=70.0, q=5.0, peak_strain=2.5e-21)` is the model of a glitch, a sine-Gaussian. `make_series(values, series)` puts an array on the time axis of the data. `highest_point(grid, phi_values)` returns the chirp mass and the $|\Phi|$ of the highest point of a profile.

We inject a sine-Gaussian at 70 Hz with $Q = 5$, 30 ms before the event, with two amplitudes: $2.5\times 10^{-21}$ and $6\times 10^{-21}$. They are large enough to change the result and too small to be obvious in the data. We redo the profile of A1 on a wider grid, from 24 to 100 solar masses. Then we subtract exactly what we injected and do it again.

In [ ]:
# the time of every sample of the data (GPS s)
t_data = np.arange(len(data)) * data.delta_t + float(data.start_time)


def sine_gaussian(t_arr, t0, f0=70.0, q=5.0, peak_strain=2.5e-21):
    """Compute a sine-Gaussian, the model of a glitch, with tau = Q / (2 pi f0).

    Args:
        t_arr: the times (s).
        t0: the central time (s).
        f0: the central frequency (Hz, default 70).
        q: the quality factor (default 5).
        peak_strain: the amplitude at the center (strain, default 2.5e-21).

    Returns:
        waveform: the sine-Gaussian at t_arr.
    """
    # the width of the Gaussian envelope
    tau = q / (2 * np.pi * f0)
    # the envelope times the oscillation
    envelope = np.exp(-((t_arr - t0) / tau) ** 2)
    carrier = np.cos(2 * np.pi * f0 * (t_arr - t0))
    waveform = peak_strain * envelope * carrier
    return waveform


def make_series(values, series):
    """Put an array on the time axis of a pycbc TimeSeries.

    Args:
        values: the samples, an array as long as series.
        series: the TimeSeries that gives the sampling and the start time.

    Returns:
        new_series: the samples as a TimeSeries.
    """
    new_series = TimeSeries(values, delta_t=series.delta_t, epoch=series.start_time)
    return new_series


def highest_point(grid, phi_values):
    """Find the highest point of a profile.

    Args:
        grid: the chirp masses (solar masses).
        phi_values: the values of |Phi| of the profile, one for each chirp mass.

    Returns:
        mass: the chirp mass at the highest point (solar masses).
        phi: the value of |Phi| there.
    """
    i = np.argmax(phi_values)
    mass = grid[i]
    phi = phi_values[i]
    return mass, phi


# a wide grid, so that a second maximum at high chirp mass is inside it
wide_grid = np.arange(24.0, 100.01, 0.5)

# the profile of the clean data and its highest point
clean = profile(data, psd, grid=wide_grid)
clean_mass, clean_phi = highest_point(wide_grid, clean)
print(f"clean data: maximum {clean_mass:.1f} at Phi {clean_phi:.1f}")

# the stretch of the grid around the clean maximum
near_clean = (wide_grid > 28) & (wide_grid < 36)

plt.figure(figsize=(8.0, 4.6))
plt.plot(wide_grid, clean, color=P['GREY_MID'], lw=2.0, label='clean data')

# a weak and a loud glitch, 30 ms before the event
peak_strains = [2.5e-21, 6e-21]
colors_g = [P['BLUE'], P['BRICK']]
names_g = ['weak glitch', 'loud glitch']
summary_lines = [f"clean: {clean_mass:.1f}, $\\Phi$ = {clean_phi:.1f}"]
for peak_strain, color, name in zip(peak_strains, colors_g, names_g):
    # the glitch and the data with the glitch in them
    glitch = sine_gaussian(t_data, GPS_GW150914 - 0.03, f0=70.0, q=5.0, peak_strain=peak_strain)
    dirty = make_series(np.asarray(data) + glitch, data)
    # the profile with the glitch
    contaminated = profile(dirty, psd, grid=wide_grid)
    # the profile after subtracting exactly the glitch that was injected
    subtracted = make_series(np.asarray(dirty) - glitch, data)
    cleaned = profile(subtracted, psd, grid=wide_grid)
    # the highest point of each profile, and the highest point around the clean maximum
    top_mass, top_phi = highest_point(wide_grid, contaminated)
    near_mass, near_phi = highest_point(wide_grid[near_clean], contaminated[near_clean])
    back_mass, back_phi = highest_point(wide_grid, cleaned)
    print(f"{name} {peak_strain:.1e}: highest point {top_mass:.1f} at Phi {top_phi:.1f}")
    print(f"    between 28 and 36 the profile runs from {contaminated[near_clean].min():.1f} "
          f"to {contaminated[near_clean].max():.1f}, highest at {near_mass:.1f} with Phi {near_phi:.1f}")
    print(f"    after subtraction: {back_mass:.1f} at Phi {back_phi:.1f}")
    # the profile with the glitch, and its highest point
    plt.plot(wide_grid, contaminated, color=color, lw=1.8, label=name)
    plt.plot(top_mass, top_phi, 'o', color=color, ms=8)
    summary_lines.append(f"{name}: {top_mass:.1f}, $\\Phi$ = {top_phi:.1f}")

plt.xlabel(r'chirp mass  [$M_\odot$], detector frame')
plt.ylabel(r'largest $|\Phi|$ near the event')
plt.ylim(0, 60)
plt.legend(loc='lower center', bbox_to_anchor=(0.5, 1.02), ncol=3, frameon=False)
plt.text(0.03, 0.96, "highest point of each profile\n" + '\n'.join(summary_lines), transform=plt.gca().transAxes,
         ha='left', va='top', fontsize=9, bbox=dict(boxstyle='round', facecolor='white', edgecolor=P['GREY_MID']))
plt.show()

**What the result shows.** With a glitch the profile has a second maximum at high chirp mass. A template of 80 solar masses is very short: it is almost a burst and it fits the glitch itself.

| | highest point | between 28 and 36 | after subtraction |
|---|---|---|---|
| clean data | 32.0, $\Phi = 19.5$ | | |
| weak glitch | 78.5, $\Phi = 21.1$ | highest at 33.5, $\Phi = 18.1$ | 32.0, $\Phi = 19.5$ |
| loud glitch | 80.5, $\Phi = 48.7$ | from 29.9 to 31.3 | 32.0, $\Phi = 19.5$ |

**The weak glitch.** A maximum remains near the true value, at 33.5 with $\Phi = 18.1$. The highest point of the profile is at 78.5 with $\Phi = 21.1$, above the 19.5 of the clean data. The highest point gives a chirp mass of 78.5 and a $\Phi$ larger than that of the clean data.

**The loud glitch.** Near the true value the profile is almost flat, with $\Phi$ between 29.9 and 31.3 over the whole stretch from 28 to 36 solar masses. The highest point is at 80.5 with $\Phi = 48.7$. The profile no longer gives the chirp mass.

Subtracting exactly what we injected gives 32.0 and 19.5 again, in both cases. In a real analysis the glitch is not known: it is fitted with its own uncertainty or the data are gated. In lab H06 a glitch gave a peak in one detector only and the coincidence of H1 and L1 rejected it.

## A4. A nested sampler on a Gaussian likelihood

The Metropolis chain of A2 gives samples and no evidence. `bilby` (Ashton and others 2019) wraps nested samplers that give both. The answer of this case is known.

We draw 100 numbers from a Gaussian of mean 3 and width 4, with the seed fixed. The class `GaussianLikelihood` is a `bilby.Likelihood`. Its method `log_likelihood` is the Gaussian log likelihood for unknown $\mu$ and $\sigma$:

$$
\ln P(\{x_{i}\}\mid\mu,\sigma) = -\frac{1}{2}\sum_{i=1}^{N}
\left[\frac{(x_{i}-\mu)^{2}}{\sigma^{2}} + \ln\left(2\pi\sigma^{2}\right)\right],
$$

with $x_{i}$ the $N = 100$ samples. The priors $p(\mu)$ and $p(\sigma)$ are flat: $\mu$ between 0 and 5 and $\sigma$ between 0 and 10. We run `dynesty` with 500 live points. We report the median and the 90 % interval of each parameter and the logarithm of the evidence $P(\{x_{i}\})$. The sample mean, the sample width and the error on the mean give the answer that is known.

In [ ]:
# the seed of the numbers and of the sampler, so that the run can be repeated
bilby.core.utils.random.seed(123)
bilby.core.utils.setup_logger(log_level="ERROR")

# 100 numbers from a Gaussian of mean 3 and width 4
x = bilby.core.utils.random.rng.normal(3.0, 4.0, 100)


class GaussianLikelihood(bilby.Likelihood):
    """The likelihood of N independent Gaussian samples with unknown mean and width."""

    def __init__(self, x):
        """Store the samples and name the two parameters of the likelihood.

        Args:
            x: the samples, an array of N numbers.
        """
        super().__init__(parameters={"mu": None, "sigma": None})
        self.x = x

    def log_likelihood(self):
        """Compute ln P(x | mu, sigma) for the values of mu and sigma that the sampler has set.

        Returns:
            log_like: the log likelihood of the samples.
        """
        mu = self.parameters["mu"]
        sigma = self.parameters["sigma"]
        # the sum of the squares of the normalized residuals
        residual = self.x - mu
        chi_square = np.sum((residual / sigma) ** 2)
        # the normalization, N ln(2 pi sigma^2)
        norm = len(self.x) * np.log(2 * np.pi * sigma ** 2)
        log_like = -0.5 * (chi_square + norm)
        return log_like


# flat priors: mu from 0 to 5 and sigma from 0 to 10
priors_gauss = dict(mu=bilby.core.prior.Uniform(0, 5, "mu"),
                    sigma=bilby.core.prior.Uniform(0, 10, "sigma"))

# nested sampling with dynesty and 500 live points
result_gauss = bilby.run_sampler(likelihood=GaussianLikelihood(x), priors=priors_gauss,
                                 sampler="dynesty", nlive=500, seed=123,
                                 outdir="outdir_h08", label="gaussian",
                                 save=False, resume=False)

# the median and the 90 % interval of each parameter
for name in ("mu", "sigma"):
    low, med, high = np.percentile(result_gauss.posterior[name], [5, 50, 95])
    print(f"{name:5s} = {med:.2f}  90 % [{low:.2f}, {high:.2f}]")

# the answer that is known: the sample mean, the sample width and the error on the mean
sample_mean = x.mean()
sample_width = x.std(ddof=1)
error_on_mean = sample_width / np.sqrt(len(x))
print(f"sample mean {sample_mean:.2f}, sample width {sample_width:.2f}, error on the mean {error_on_mean:.2f}")

# the logarithm of the evidence, with its error
print(f"ln P(x) = {result_gauss.log_evidence:.2f} +- {result_gauss.log_evidence_err:.2f}")

# the two posteriors, with the true value (dotted) and the sample value (dashed)
fig, axes = plt.subplots(1, 2, figsize=(9, 3.6))
for ax, name, true_value, estimate in zip(axes, ("mu", "sigma"), (3.0, 4.0), (sample_mean, sample_width)):
    ax.hist(result_gauss.posterior[name], bins=30, density=True, color=P['BLUE'], alpha=0.6)
    ax.axvline(true_value, color=P['GREY_DARK'], ls=':', lw=1.5, label='true value')
    ax.axvline(estimate, color=P['BRICK'], ls='--', lw=1.5, label='sample value')
    ax.set_xlabel(r'$\mu$' if name == "mu" else r'$\sigma$')
    ax.set_ylabel('posterior density')
handles, labels = axes[0].get_legend_handles_labels()
fig.legend(handles, labels, frameon=False, loc='upper center', ncol=2)
plt.tight_layout(rect=(0, 0, 1, 0.92))
plt.show()

**What the result shows.** The run takes about fifteen seconds. The posterior gives $\mu = 3.32$ with 90 % in $[2.70, 3.92]$ and $\sigma = 3.63$ in $[3.25, 4.11]$. The sample mean is 3.33, the sample width is 3.60 and the error on the mean is 0.36.

The half-width of the 90 % interval of $\mu$ is 0.61, which is $1.7$ times the error on the mean: the Gaussian interval, as it must be for a flat prior and a likelihood this regular. The true mean (3) and the true width (4) are inside their intervals. In the plot the posterior is centered on the dashed line, the sample value.

The sampler also returns $\ln P(\{x_{i}\}) = -273.85 \pm 0.11$. The number has no meaning by itself. B6 compares it with the evidence of a model with $\sigma$ fixed.

## A5. Nested sampling on an injected binary

We simulate the H1 and L1 noise from their design spectra with `set_strain_data_from_power_spectral_densities`, four seconds at 2048 Hz. We inject a GW150914-like signal: chirp mass $28.1\,M_{\odot}$, mass ratio $29/36$, no spins, 1500 Mpc, `IMRPhenomPv2` from 20 Hz. **This noise is simulated**: Gaussian and stationary, with no glitch and no line.

We fix every parameter at its injected value except three: the chirp mass, the mass ratio and the distance $r$, which `bilby` calls `luminosity_distance`. The priors are `UniformInComponentsChirpMass` (20 to 35), `UniformInComponentsMassRatio` (0.4 to 1) and `UniformSourceFrame` (200 to 5000 Mpc). We run `dynesty` with 100 live points and the `acceptance-walk` move, the light configuration: about two minutes on one core. The posterior is $P(\boldsymbol{\theta}\mid s) = P(s\mid\boldsymbol{\theta})\,p(\boldsymbol{\theta})/P(s\mid h)$ of lecture 16 and the sampler returns it with the evidence $P(s\mid h)$.

`interval_text(samples)` writes the median and the 90 % interval of a parameter as one string. We report the intervals, the correlation of $m_{1}$ with $m_{2}$ and the Bayes factor against noise, $\mathrm{BF} = P(s\mid h)/P(s\mid 0)$. Then we draw the corner plot.

In [ ]:
def interval_text(samples):
    """Write the median and the 90 % interval of a set of samples as one string.

    Args:
        samples: the posterior samples of one parameter, an array.

    Returns:
        text: the median and the interval, as 'median [low, high]'.
    """
    low, med, high = np.percentile(samples, [5, 50, 95])
    text = f"{med:8.2f} [{low:8.2f}, {high:8.2f}]"
    return text


# the seed of the noise and of the sampler, so that the run can be repeated
bilby.core.utils.random.seed(88)
bilby.core.utils.setup_logger(log_level="ERROR")

# four seconds of data at 2048 Hz
duration = 4.0
sampling_frequency = 2048.0

# the injected GW150914-like signal: masses, spins, distance, angles and time
injection = dict(chirp_mass=28.1, mass_ratio=29 / 36, a_1=0.0, a_2=0.0,
                 tilt_1=0.0, tilt_2=0.0, phi_12=0.0, phi_jl=0.0,
                 luminosity_distance=1500.0, theta_jn=0.4, psi=2.659,
                 phase=1.3, ra=1.375, dec=-1.2108, geocent_time=1126259462.4)

# the waveform generator: IMRPhenomPv2 from 20 Hz
generator = bilby.gw.waveform_generator.WaveformGenerator(
    duration=duration, sampling_frequency=sampling_frequency,
    frequency_domain_source_model=bilby.gw.source.lal_binary_black_hole,
    parameter_conversion=bilby.gw.conversion.convert_to_lal_binary_black_hole_parameters,
    waveform_arguments=dict(waveform_approximant="IMRPhenomPv2",
                            reference_frequency=50.0, minimum_frequency=20.0))

# simulated noise of H1 and L1 from their design spectra, with the signal injected in it
ifos = bilby.gw.detector.InterferometerList(["H1", "L1"])
ifos.set_strain_data_from_power_spectral_densities(
    sampling_frequency=sampling_frequency, duration=duration,
    start_time=injection["geocent_time"] - 3)
ifos.inject_signal(waveform_generator=generator, parameters=injection)

# the optimal SNR of the injection in each detector
for ifo in ifos:
    print(f"{ifo.name}: optimal SNR {abs(ifo.meta_data['optimal_SNR']):.1f}")

# the priors: everything fixed at the injected value except three parameters
priors_binary = bilby.gw.prior.BBHPriorDict(injection.copy())
priors_binary["chirp_mass"] = bilby.gw.prior.UniformInComponentsChirpMass(
    20.0, 35.0, name="chirp_mass")
priors_binary["mass_ratio"] = bilby.gw.prior.UniformInComponentsMassRatio(
    0.4, 1.0, name="mass_ratio")
priors_binary["luminosity_distance"] = bilby.gw.prior.UniformSourceFrame(
    200.0, 5000.0, name="luminosity_distance")

# the likelihood of the two detectors
likelihood = bilby.gw.likelihood.GravitationalWaveTransient(
    interferometers=ifos, waveform_generator=generator)

# nested sampling: 100 live points, and the walk that moves them
# (a full analysis uses more live points and a longer walk)
result_light = bilby.run_sampler(likelihood=likelihood, priors=priors_binary,
                                 sampler="dynesty", nlive=100, seed=88,
                                 sample="acceptance-walk", naccept=10, walks=20,
                                 injection_parameters=injection,
                                 outdir="outdir_h08", label="injection",
                                 save=False, resume=False)

# the posterior samples, with the two component masses added
post_light = bilby.gw.conversion.generate_mass_parameters(result_light.posterior)

# the median and the 90 % interval of each parameter, with the injected value where there is one
names = ["chirp_mass", "mass_ratio", "mass_1", "mass_2", "luminosity_distance"]
for name in names:
    if name in injection:
        tag = f"(injected {injection[name]:.2f})"
    else:
        tag = ""
    print(f"{name:20s} {interval_text(post_light[name])} {tag}")

# the Bayes factor against noise, and the correlation of the two component masses
corr_masses = np.corrcoef(post_light['mass_1'], post_light['mass_2'])[0, 1]
print(f"ln Bayes factor against noise {result_light.log_bayes_factor:.1f}")
print(f"correlation of m1 and m2: {corr_masses:+.2f}")

# the corner plot of the three free parameters, with the injected values as lines and the distance written r
shown = dict(chirp_mass=28.1, mass_ratio=29 / 36, luminosity_distance=1500.0)
labels = result_light.get_latex_labels_from_parameter_keys(list(shown))
labels[-1] = r"$r$ [Mpc]"
fig = result_light.plot_corner(parameters=shown, labels=labels, quantiles=[0.05, 0.95], save=False)
plt.show()

**What the result shows.** The optimal SNRs are 14.7 in H1 and 11.9 in L1. The posterior gives

| parameter | median | 90 % interval | injected |
|---|---|---|---|
| chirp mass | 28.23 | 28.02 to 28.41 | 28.10 |
| mass ratio | 0.83 | 0.80 to 0.87 | 0.81 |
| $m_{1}$, $m_{2}$ | 35.6, 29.6 | 35.0 to 36.1, 28.8 to 30.4 | 36, 29 |
| distance (Mpc) | 1343 | 1249 to 1455 | 1500 |

and $\ln\mathrm{BF} = 216.7$ against noise.

The chirp mass is measured to 0.7 % (half of the width of the interval), as lecture 16 said it would be. The component masses anticorrelate at $-0.95$: the corner plot shows a thin ridge of constant chirp mass. The mass ratio is narrow here only because the spins are held at zero.

The injected distance is outside its 90 % interval. With the inclination held fixed the distance follows the measured amplitude. This noise draw made the signal look louder than it is. A 90 % interval misses the true value one time in ten. Its width, about $\pm 7$ %, is the statistical width of the distance in this run. This run uses the exact spectrum and has no glitch and no calibration error.

## Part B. Your turn: instrumental effects and the sampler

Part A gave the statistical widths and showed a glitch. In Part B we measure the effect of other instrumental choices and compare each with the width. These are the functions we have defined for you. Twelve are functions of Part A and one is new. Call them with keywords, as in Part A.

- `load_h1(path=H1_FILE, f_high_pass=15.0, fs_out=2048.0)` returns the strain of H1 as a `pycbc` time series.
- `condition(series, edge=2, seglen=4)` returns the cropped data and the Welch estimate of $S_{n}(f)$ on segments of `seglen` seconds.
- `template_for(chirp_mass, series, f_low=20.0)` returns the equal mass template of a given chirp mass.
- `profile(series, psd, grid, t_event=GPS_GW150914, half=0.1, f_low=20.0, crop=(6, 4))` returns the largest $|\Phi|$ near the event for each chirp mass of `grid`.
- `sine_gaussian(t_arr, t0, f0=70.0, q=5.0, peak_strain=2.5e-21)` returns a glitch at the times `t_arr`. The array `t_data` holds the times of the samples of `data`.
- `make_series(values, series)` puts an array on the time axis of `series` and returns a time series.
- `highest_point(grid, phi_values)` returns the chirp mass and the $|\Phi|$ of the highest point of a profile.
- `signal_amplitude(distance, iota, f_plus=1.0, f_cross=1.0)` returns the amplitude that a detector measures, up to a constant.
- `make_detector(f_plus=1.0, f_cross=1.0, distance=R_TRUE, iota=IOTA_TRUE)` returns the tuple (measured amplitude, `f_plus`, `f_cross`) of a detector.
- `log_posterior(distance, iota, prior='volume', detectors=ONE_DETECTOR, error=FRACTIONAL_ERROR)` returns $\ln P(r,\iota\mid A)$ up to a constant.
- `sample(prior='volume', detectors=ONE_DETECTOR, steps=60000, step_size=0.08, seed=4)` returns the samples of the Metropolis chain: an array with the columns $r$ and $\iota$.
- `interval_text(samples)` returns the median and the 90 % interval as a string.
- `psd_from_stretch(series, t_start, t_stop, seglen=4)` is the new one. It returns $S_{n}(f)$ estimated only on the data between the GPS times `t_start` and `t_stop`, on the frequency grid of `series`.

**What we have from Part A.** `raw`, `data`, `psd`, `grid`, `peak_mass`, `t_data`, `wide_grid`, `x`, `GaussianLikelihood`, `priors_gauss`, `result_gauss`, `injection`, `likelihood`, `priors_binary`, `result_light` and `post_light`.

**The statistical widths.** The 90 % interval of the distance in A2 and the 90 % intervals of A5 are the widths. A shift that is smaller than its width does not change the result. A shift of the same size does.

In [ ]:
# the new function of Part B
def psd_from_stretch(series, t_start, t_stop, seglen=4):
    """Estimate the noise spectrum on a stretch of the data only.

    Args:
        series: the data, a pycbc TimeSeries.
        t_start: the GPS time where the stretch starts (s).
        t_stop: the GPS time where the stretch ends (s).
        seglen: the length of the Welch segments (s, default 4).

    Returns:
        stretch_psd: the one-sided PSD S_n(f) of the stretch, on the frequency grid of series.
    """
    # the stretch of the data between the two times
    stretch = series.time_slice(t_start, t_stop)
    # Welch estimate on segments of seglen seconds, then on the frequency grid of the whole data
    stretch_psd = interpolate(stretch.psd(seglen), series.delta_f)
    return stretch_psd

### B1. The estimate of the noise spectrum

The likelihood contains the one-sided PSD $S_{n}(f)$, which is estimated from the same data. Estimate the spectrum on segments of 1, 2, 4 and 8 seconds. For each spectrum report $\Phi$ at the peak chirp mass of A1 and the distance it implies relative to the 4 s case.

What to look at: the spread of $\Phi$ across the four choices. $\Phi$ sets the amplitude and the amplitude sets the distance, so a change of one percent in $\Phi$ is a change of one percent in every distance derived from it.

*Hint:* `condition(raw, seglen=...)` returns the data and the spectrum. `profile(data_seg, psd_seg, grid=np.array([peak_mass]))` gives $\Phi$ at one chirp mass. The distance scales as $1/\Phi$.

In [ ]:
# your code here

> **Solution**
>
> Worked through in the session. Write your own version first — comparing
> the two is where the value is.


### B2. The noise spectrum from data far from the event

Estimate the spectrum on a stretch of data far from the event and use it for the whole record. Use the first 10 s of `data` and then the last 10 s. For each spectrum report $\Phi$ at the peak chirp mass of A1. Compare the spread with the one of B1: which effect is larger?

*Hint:* `psd_from_stretch(data, t_start, t_stop)` gives the spectrum of a stretch. The start of the data is `float(data.start_time)` and the end is `float(data.end_time)`. `profile(data, psd_far, grid=np.array([peak_mass]))` gives $\Phi$.

In [ ]:
# your code here

> **Solution**
>
> Worked through in the session. Write your own version first — comparing
> the two is where the value is.


### B3. The calibration

A gravitational wave detector measures a strain only after a calibration. The calibration has an uncertainty of a few percent in amplitude. Multiply the data by $1 \pm 0.05$ and rerun the filter at the peak chirp mass. Report $\Phi$ and the distance it implies relative to the uncalibrated case.

What to look at: the propagation of the error. Which parameters change and which do not.

*Hint:* `data * (1 + epsilon)` is the scaled data. Use the spectrum `psd` of A1 for all three.

In [ ]:
# your code here

> **Solution**
>
> Worked through in the session. Write your own version first — comparing
> the two is where the value is.


### B4. How far from the merger a glitch can be and be ignored

Inject the weak glitch of A3 ($2.5\times 10^{-21}$, 70 Hz, $Q = 5$) at 0.03, 0.1, 0.3 and 1 s before the merger. For each time compute the profile of A1 on `grid` and print the chirp mass and the $\Phi$ of its highest point. Find the separation at which the shift of the peak is not more than one step of the grid.

*Optional:* repeat with a glitch of 150 Hz. Does the answer depend on the frequency of the glitch?

*Hint:* `sine_gaussian(t_data, GPS_GW150914 - offset, f0=..., peak_strain=...)` is the glitch. Add it with `make_series(np.asarray(data) + glitch, data)`. `highest_point(grid, ...)` gives the peak.

In [ ]:
# your code here

> **Solution**
>
> Worked through in the session. Write your own version first — comparing
> the two is where the value is.


### B5. A second detector and the degeneracy

Repeat A2 with two amplitudes. The first detector has the responses 1 and 1 as in A2. The second has $F_{+} = 0.3$ and $F_{\times} = 0.9$ and measures the amplitude of the same source. Sample the posterior with the volume prior and compare the 90 % interval of $r$ and the correlation with $\cos\iota$ with the one-detector result of A2.

What to look at: the narrowing of the interval and the change of the correlation.

*Hint:* `make_detector(f_plus=0.3, f_cross=0.9)` returns the second detector. Pass `detectors=[first, second]` to `sample`.

In [ ]:
# your code here

> **Solution**
>
> Worked through in the session. Write your own version first — comparing
> the two is where the value is.


### B6. The Gaussian model with a fixed width

Run the nested sampler of A4 again with $\sigma$ fixed at 4, the true value. The prior of $\sigma$ becomes a number and the sampler has one free parameter. Print the median and the 90 % interval of $\mu$ and $\ln P(\{x_{i}\})$. The difference of the two $\ln P(\{x_{i}\})$ is the logarithm of the Bayes factor of the model with free $\sigma$ against the model with $\sigma = 4$.

*Hint:* make `priors_fixed = bilby.core.prior.PriorDict(dict(priors_gauss))` and set `priors_fixed["sigma"] = 4.0`. Give the run a new `label`. With one free parameter the default move of `dynesty` does not finish: use `sample="acceptance-walk", naccept=10, walks=20` as in A5.

In [ ]:
# your code here

> **Solution**
>
> Worked through in the session. Write your own version first — comparing
> the two is where the value is.


### B7. Optional: the full configuration of the sampler

Run A5 again with the full configuration: the same data, model and priors, with 250 live points and the default walk of `dynesty`. Leave out `sample`, `naccept` and `walks` and give the run a new `label`, so that it does not reuse the light result. It takes three to four minutes.

Then print the medians and the 90 % intervals of the two runs side by side, with the two Bayes factors. Draw the two posteriors of the chirp mass on one figure.

What to look at: which numbers change with more live points and which do not.

*Hint:* `bilby.run_sampler(likelihood=likelihood, priors=priors_binary, sampler="dynesty", nlive=250, seed=88, injection_parameters=injection, outdir="outdir_h08", label="injection_full", save=False, resume=False)`. Then `bilby.gw.conversion.generate_mass_parameters(result_full.posterior)` and `interval_text`.

In [ ]:
# your code here

> **Solution**
>
> Worked through in the session. Write your own version first — comparing
> the two is where the value is.


![Corner plot of the full run: chirp mass, mass ratio and distance, with the injected values as orange lines and the 90 % intervals as dashed lines.](figures/h08_full_run_corner.png)

The reference result of the full run, with 250 live points and the default sampler setting, for comparison if your run is still going.

## Software

`pycbc` for the filter and the spectrum, a Metropolis chain written in A2 for the sampling. A4, A5, B6 and B7 use a nested sampler through `bilby` (Ashton and others 2019), which returns the evidence as well as the samples. With three free parameters it takes minutes. A real analysis frees fifteen and takes hours per event.

The glitch in A3 and B4 and the noise of A5 and B7 are **simulated**. The glitch that was present in GW170817 was real and the analysis that removed it is described in LIGO Scientific and Virgo Collaborations (2019).

**References**

- Ashton and others (2019). *BILBY: A user-friendly Bayesian inference library for gravitational-wave astronomy*.
- Cutler and Flanagan (1994). *Gravitational waves from merging compact binaries: How accurately can one extract the binary's parameters from the inspiral waveform?*.
- LIGO Scientific Collaboration and Virgo Collaboration (2017). *GW170817: Observation of Gravitational Waves from a Binary Neutron Star Inspiral*.
- LIGO Scientific and Virgo Collaborations (2019). *Properties of the Binary Neutron Star Merger GW170817*.